In [1]:
import wandb
import re
import pandas as pd

In [29]:
!wandb login --relogin 320a98c6a9ba9e09b5d88d9b64059d763ae265f4

wandb: Appending key for api.wandb.ai to your netrc file: /home/influx/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


In [30]:
api = wandb.Api()

In [35]:
runs = api.runs("influx_m/RL-DOW30-4SUB-final")

In [36]:
# Only keep these fields
wanted_fields = [
    "Start",
    "End",
    "Annualized Return [%]",
    "Annualized Volatility [%]",
    "Max Drawdown [%]",
    "Average Drawdown [%]",
    "Sharpe Ratio",
    "Sortino Ratio",
    "Calmar Ratio",
    "Max Drawdown Duration (days)",
    "Number of Trades"
]

In [67]:
def summarize_runs(pattern: str):
    regex = re.compile(pattern)

    data = []
    for run in runs:
        if regex.match(run.name):
            res_trade = run.summary.get("res_trade", {})
            trade_metrics = run.summary.get("trade_metrics", {})
            row = {field: res_trade.get(field) for field in wanted_fields if field in res_trade}
            row["Number of Trades"] = trade_metrics.get("Number of Trades")
            data.append(row)

    df = pd.DataFrame(data)

    if df.empty:
        print("No runs matched the pattern.")
        return

    means = df.mean(numeric_only=True)
    stds = df.std(numeric_only=True)

    for field in wanted_fields:
        if field in df.columns:
            if pd.api.types.is_numeric_dtype(df[field]):
                print(f"{field}: {means[field]:.2f} ± {stds[field]:.2f}")
            else:
                print(f"{field}: {df[field].unique()}")

    # Add number of runs matched
    print(f"\nTotal runs matched: {len(data)}")


In [68]:
summarize_runs(r"^init ppo .* 1 False$")

Start: ['2023-01-03T00:00:00']
End: ['2025-08-29T00:00:00']
Annualized Return [%]: 25.65 ± 4.47
Annualized Volatility [%]: 20.46 ± 1.07
Max Drawdown [%]: -24.78 ± 1.82
Average Drawdown [%]: -5.33 ± 0.57
Sharpe Ratio: 1.22 ± 0.19
Sortino Ratio: 1.58 ± 0.24
Calmar Ratio: 1.04 ± 0.19
Max Drawdown Duration (days): 127.60 ± 19.68
Number of Trades: 122.50 ± 58.41

Total runs matched: 10


In [69]:
summarize_runs(r"^init ppo .* 1 True$")

Start: ['2023-01-03T00:00:00']
End: ['2025-08-29T00:00:00']
Annualized Return [%]: 21.35 ± 4.97
Annualized Volatility [%]: 19.25 ± 2.13
Max Drawdown [%]: -23.43 ± 1.94
Average Drawdown [%]: -6.05 ± 1.58
Sharpe Ratio: 1.11 ± 0.22
Sortino Ratio: 1.48 ± 0.29
Calmar Ratio: 0.91 ± 0.20
Max Drawdown Duration (days): 182.70 ± 88.75
Number of Trades: 124.70 ± 47.89

Total runs matched: 10


In [70]:
summarize_runs(r"^init ppo .* 3 False$")

Start: ['2023-01-03T00:00:00']
End: ['2025-08-29T00:00:00']
Annualized Return [%]: 19.10 ± 3.98
Annualized Volatility [%]: 21.25 ± 1.35
Max Drawdown [%]: -24.33 ± 2.20
Average Drawdown [%]: -7.79 ± 1.21
Sharpe Ratio: 0.93 ± 0.14
Sortino Ratio: 1.24 ± 0.18
Calmar Ratio: 0.79 ± 0.18
Max Drawdown Duration (days): 250.20 ± 108.01
Number of Trades: 139.60 ± 11.89

Total runs matched: 5


In [71]:
summarize_runs(r"^init ppo .* 3 True$")

Start: ['2023-01-03T00:00:00']
End: ['2025-08-29T00:00:00']
Annualized Return [%]: 20.89 ± 4.75
Annualized Volatility [%]: 19.21 ± 2.86
Max Drawdown [%]: -23.24 ± 2.86
Average Drawdown [%]: -5.62 ± 1.48
Sharpe Ratio: 1.09 ± 0.16
Sortino Ratio: 1.45 ± 0.20
Calmar Ratio: 0.90 ± 0.15
Max Drawdown Duration (days): 154.25 ± 69.62
Number of Trades: 154.00 ± 33.93

Total runs matched: 4


In [72]:
summarize_runs(r"^init ppo .* 1 False risk-reward$")

Start: ['2023-01-03T00:00:00']
End: ['2025-08-29T00:00:00']
Annualized Return [%]: 24.38 ± 8.64
Annualized Volatility [%]: 20.52 ± 1.45
Max Drawdown [%]: -25.03 ± 0.82
Average Drawdown [%]: -6.26 ± 2.13
Sharpe Ratio: 1.16 ± 0.35
Sortino Ratio: 1.55 ± 0.47
Calmar Ratio: 0.97 ± 0.33
Max Drawdown Duration (days): 192.00 ± 113.68
Number of Trades: 127.80 ± 47.99

Total runs matched: 5


In [73]:
summarize_runs(r"^init ppo .* 3 False risk-reward$")

Start: ['2023-01-03T00:00:00' nan]
End: ['2025-08-29T00:00:00' nan]
Annualized Return [%]: 21.97 ± 8.12
Annualized Volatility [%]: 20.84 ± 1.39
Max Drawdown [%]: -25.33 ± 0.92
Average Drawdown [%]: -6.44 ± 2.55
Sharpe Ratio: 1.05 ± 0.31
Sortino Ratio: 1.40 ± 0.40
Calmar Ratio: 0.86 ± 0.30
Max Drawdown Duration (days): 214.50 ± 134.69
Number of Trades: 135.75 ± 38.51

Total runs matched: 5


In [74]:
summarize_runs(r"^init rppo .* 1 False$")

Start: ['2023-01-03T00:00:00']
End: ['2025-08-29T00:00:00']
Annualized Return [%]: 22.67 ± 5.21
Annualized Volatility [%]: 19.68 ± 1.73
Max Drawdown [%]: -24.53 ± 1.58
Average Drawdown [%]: -5.85 ± 1.62
Sharpe Ratio: 1.14 ± 0.23
Sortino Ratio: 1.53 ± 0.31
Calmar Ratio: 0.93 ± 0.21
Max Drawdown Duration (days): 175.00 ± 99.07
Number of Trades: 85.30 ± 30.01

Total runs matched: 10


In [75]:
summarize_runs(r"^init rppo .* 1 True$")

Start: ['2023-01-03T00:00:00' nan]
End: ['2025-08-29T00:00:00' nan]
Annualized Return [%]: 22.05 ± 3.40
Annualized Volatility [%]: 19.05 ± 1.94
Max Drawdown [%]: -24.13 ± 1.45
Average Drawdown [%]: -5.57 ± 1.36
Sharpe Ratio: 1.16 ± 0.20
Sortino Ratio: 1.56 ± 0.26
Calmar Ratio: 0.92 ± 0.16
Max Drawdown Duration (days): 142.33 ± 58.01
Number of Trades: 112.17 ± 35.53

Total runs matched: 7


In [76]:
summarize_runs(r"^init rppo .* 3 False$")

Start: ['2023-01-03T00:00:00']
End: ['2025-08-29T00:00:00']
Annualized Return [%]: 28.62 ± 2.23
Annualized Volatility [%]: 19.60 ± 1.69
Max Drawdown [%]: -25.11 ± 2.97
Average Drawdown [%]: -4.48 ± 0.40
Sharpe Ratio: 1.39 ± 0.04
Sortino Ratio: 1.83 ± 0.11
Calmar Ratio: 1.14 ± 0.06
Max Drawdown Duration (days): 99.80 ± 8.84
Number of Trades: 89.80 ± 36.19

Total runs matched: 5


In [77]:
summarize_runs(r"init rppo .* 3 True$")

Start: ['2023-01-03T00:00:00' nan]
End: ['2025-08-29T00:00:00' nan]
Annualized Return [%]: 25.26 ± 6.75
Annualized Volatility [%]: 19.52 ± 1.06
Max Drawdown [%]: -24.32 ± 1.56
Average Drawdown [%]: -5.24 ± 1.89
Sharpe Ratio: 1.26 ± 0.33
Sortino Ratio: 1.71 ± 0.46
Calmar Ratio: 1.04 ± 0.31
Max Drawdown Duration (days): 146.00 ± 94.41
Number of Trades: 75.00 ± 65.87

Total runs matched: 4


In [78]:
summarize_runs(r"^init rppo .* 1 False risk-reward$")

Start: ['2023-01-03T00:00:00']
End: ['2025-08-29T00:00:00']
Annualized Return [%]: 25.47 ± 4.32
Annualized Volatility [%]: 20.18 ± 1.94
Max Drawdown [%]: -25.33 ± 2.01
Average Drawdown [%]: -5.21 ± 0.72
Sharpe Ratio: 1.23 ± 0.17
Sortino Ratio: 1.64 ± 0.26
Calmar Ratio: 1.00 ± 0.11
Max Drawdown Duration (days): 120.60 ± 24.14
Number of Trades: 85.80 ± 39.42

Total runs matched: 5


In [79]:
summarize_runs(r"^init rppo .* 3 False risk-reward$")

Start: ['2023-01-03T00:00:00' nan]
End: ['2025-08-29T00:00:00' nan]
Annualized Return [%]: 27.01 ± 3.13
Annualized Volatility [%]: 19.48 ± 1.33
Max Drawdown [%]: -25.46 ± 2.07
Average Drawdown [%]: -4.90 ± 0.68
Sharpe Ratio: 1.33 ± 0.12
Sortino Ratio: 1.79 ± 0.18
Calmar Ratio: 1.06 ± 0.04
Max Drawdown Duration (days): 117.25 ± 15.95
Number of Trades: 119.25 ± 31.69

Total runs matched: 5
